# Random-Wave Line Scattering Demo

This notebook demonstrates `rw_line_scattering.py`, which estimates the line-density correlation for the line set from two independent monochromatic Gaussian random waves and computes the isotropic scattering spectrum.

The demo parameters below use a higher-r grid than the first quick pass, because the Fourier integral needs a long real-space tail to behave well at low Q. Increase `Nr` and `N_samp` further for production-quality curves.

In [ ]:
%matplotlib inline

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, display
from scipy.integrate import simpson

import rw_line_scattering as ris

plt.rcParams.update({
    "figure.figsize": (7, 4.5),
    "axes.grid": True,
    "grid.alpha": 0.25,
})

## Parameters

In [ ]:
k0 = 1.0
r_min = 2e-3 / k0
r_max = 5e2 / k0
Nr = 5000
# r_split = 2.0 / k0
# Nr_small = 500

Q_min = 0.1 * k0
Q_max = 20 * k0
NQ = 1001
tail_start = 0.8 * r_max
use_singular_split = False
rc = 0.5 / k0
taper_start_fractions = [0.60, 0.75, 0.90]
r_max_convergence_values = [100/k0, 200/k0, 300/k0]

N_samp = 2**15
use_qmc = True
random_seed = 12345

a = k0**2 / 3.0
rho0 = a / np.pi

# r_grid = ris.make_r_grid(r_min, r_max, Nr, mode="linear", r_split=r_split, n_small=Nr_small)
r_grid = np.linspace(r_min, r_max, Nr)
Q_grid = np.logspace(np.log10(Q_min), np.log10(Q_max), NQ)

print(f"a = {a:.12g}")
print(f"rho0 = {rho0:.12g}")
print(f"rho0^2 background plateau = {rho0**2:.12g}")
print(f"use_singular_split = {use_singular_split}")
print(f"singular split rc = {rc:.12g}")
print(f"r grid: {len(r_grid)} points; min dr = {np.min(np.diff(r_grid)):.4g}; max dr = {np.max(np.diff(r_grid)):.4g}")
print(f"expected M_J(0+) = {2*a*a:.12g}")
print(f"expected M_J(infinity) = {4*a*a:.12g}")
print(f"expected Q*I(Q) plateau = {k0**2/3:.12g}")

## Covariance And Derivatives

In [ ]:
fig, ax = plt.subplots()
ax.plot(r_grid, ris.g_mono(r_grid, k0), label="g(r)")
ax.plot(r_grid, ris.gp_mono(r_grid, k0), label="g'(r)")
ax.plot(r_grid, ris.gpp_mono(r_grid, k0), label="g''(r)")
ax.set_xlabel("r")
ax.legend()
plt.show()

## Estimate The Line-Density Correlation

In [ ]:
M_J, C_L = ris.compute_CL(
    r_grid,
    k0,
    N_samp,
    use_qmc=use_qmc,
    random_seed=random_seed,
    progress=True,
)

print(f"empirical M_J(r_min={r_grid[0]:.6g}) = {M_J[0]:.12g}")
print(f"empirical M_J(r_max={r_grid[-1]:.6g}) = {M_J[-1]:.12g}")

In [ ]:
fig, ax = plt.subplots()
ax.plot(r_grid, M_J, label="M_J(r)")
ax.axhline(2*a*a, color="tab:orange", linestyle="--", label="2 a^2")
ax.axhline(4*a*a, color="tab:green", linestyle="--", label="4 a^2")
ax.set_xlabel("r")
ax.set_xscale("log")
ax.set_ylabel("M_J")
ax.legend()
plt.show()

In [ ]:
C_L_coherent = ris.coherent_CL(C_L, k0)

fig, ax = plt.subplots()
ax.loglog(r_grid, C_L, label="C_L(r)")
ax.loglog(r_grid, rho0/(2*np.pi*r_grid**2), "--", label="rho0/(2 pi r^2)")
ax.axhline(rho0**2, color="tab:green", linestyle=":", label="rho0^2")
ax.set_xlabel("r")
ax.set_ylabel("C_L")
ax.legend()
plt.show()

In [ ]:
fig, ax = plt.subplots()
ax.semilogx(r_grid, C_L_coherent, label="C_L(r) - rho0^2")
ax.axhline(0.0, color="black", linewidth=0.8)
ax.set_xlabel("r")
ax.set_ylabel("coherent C_L")
ax.legend()
plt.show()

## Compute The Isotropic Scattering Spectrum

For this no-split test, subtract the background `rho0^2` plateau and directly transform the coherent correlation on the sampled grid.

In [ ]:
diag = ris.compute_transform_diagnostics(
    r_grid,
    C_L,
    Q_grid,
    k0,
    r_taper_start=tail_start,
    use_singular_split=use_singular_split,
    rc=rc,
)

W_tail = diag["w_tail"]
C_L_coherent = diag["CL_coherent"]
C_L_sing = diag["CL_sing"]
C_L_reg = diag["CL_reg"]
C_L_transform_raw = C_L_coherent
C_L_transform = C_L_reg * W_tail if use_singular_split else C_L_coherent * W_tail

I_Q_full_direct = diag["I_full_windowed"]
I_plateau_windowed = diag["I_plateau_windowed"]
I_Q_raw = diag["I_coherent_windowed"]
I_sing = diag["I_sing"]
I_reg_windowed = diag["I_reg_windowed"]
I_Q = diag["I_total"]

high_q = (Q_grid/k0 >= 5.0) & (Q_grid/k0 <= 50.0)
plateau_median = np.median(Q_grid[high_q] * I_Q[high_q])
print(f"tail window starts at r = {tail_start:.6g}")
print(f"CL[-1] = {C_L[-1]:.12g}")
print(f"CL[-1] - rho0^2 = {C_L[-1] - rho0**2:.12g}")
print(f"median Q*I(Q), Q/k0 in [5, 50] = {plateau_median:.12g}")

In [ ]:
fig, ax = plt.subplots()
ax.plot(Q_grid, I_Q, '.-', linewidth=2.2, label="bg + window")
# ax.loglog(Q_grid, I_Q_raw, alpha=0.55, label="bg subtracted direct raw")
# ax.loglog(Q_grid, I_Q_full_direct, alpha=0.35, label="full C_L direct")
ax.plot(Q_grid, 1/(3*Q_grid), "--", label="1/(3Q)")
ax.set_xlabel("Q")
ax.set_ylabel("I(Q)")
ax.set_xscale("log")
ax.set_yscale("log")
ax.legend()
plt.show()

In [ ]:
if use_singular_split:
    fig, ax = plt.subplots()
    ax.loglog(r_grid, C_L, label="C_L")
    ax.loglog(r_grid, C_L_sing, "--", label="C_sing")
    ax.loglog(r_grid, np.abs(C_L_reg), alpha=0.75, label="|C_reg|")
    ax.set_xlabel("r")
    ax.set_ylabel("correlation")
    ax.legend()
    plt.show()

fig, ax = plt.subplots()
ax.plot(r_grid, W_tail, label="tail window")
ax.axvline(tail_start, color="tab:orange", linestyle="--", label="taper start")
ax.set_xlabel("r")
ax.set_ylabel("w_tail")
ax.legend()
plt.show()

fig, ax = plt.subplots()
ax.loglog(Q_grid/k0, np.abs(I_Q_full_direct), label="|H[C_L*w]| diagnostic")
ax.loglog(Q_grid/k0, np.abs(I_plateau_windowed), label="|H[rho0^2*w]| leakage")
ax.loglog(Q_grid/k0, np.abs(I_Q_raw), label="|H[(C_L-rho0^2)*w]|")
ax.set_xlabel("Q/k0")
ax.set_ylabel("absolute intensity")
ax.legend()
plt.show()

In [ ]:
fig, ax = plt.subplots()
ax.semilogx(Q_grid, Q_grid * I_Q, linewidth=2.2, label="coherent direct + tail window")
ax.semilogx(Q_grid, Q_grid * I_Q_raw, alpha=0.55, label="coherent direct raw")
# ax.semilogx(Q_grid, Q_grid * I_Q_full_direct, alpha=0.35, label="full C_L direct")
ax.axhline(k0**2/3.0, color="tab:orange", linestyle="--", label="k0^2/3")
ax.set_xlabel("Q")
ax.set_ylabel("Q I(Q)")
ax.legend()
plt.show()

## Q/k0 = 1 Tail Diagnostics

These diagnostics test whether the feature near `Q/k0 = 1` is coming from the coherent real-space tail, the transform window, finite `r_max`, covariance ingredients, or `M_J(r)` itself.

In [ ]:
output_dir = Path("rw_line_scattering_demo_output")
output_dir.mkdir(exist_ok=True)

r_tail_min = 20.0 / k0
r_tail_max = r_max
tail_projection_coherent = ris.tail_frequency_projection(
    r_grid,
    C_L_coherent,
    k0,
    r_tail_min=r_tail_min,
    r_tail_max=r_tail_max,
)

print("C_coherent tail projection")
print(f"  RelAmp1 @ k0   = {tail_projection_coherent['RelAmp1']:.6g}")
print(f"  RelAmp2 @ 2*k0 = {tail_projection_coherent['RelAmp2']:.6g}")

fig, ax = plt.subplots()
ax.bar(["k0", "2 k0"], [tail_projection_coherent["RelAmp1"], tail_projection_coherent["RelAmp2"]])
ax.set_ylabel("relative tail projection amplitude")
ax.set_title("C_coherent tail frequency projection")
fig.tight_layout()
fig.savefig(output_dir / "diagnostics_tail_frequency_projection.png", dpi=180)
plt.show()

fig, ax = plt.subplots()
ax.plot(r_grid*k0, C_L, label="C_L")
ax.axhline(rho0**2, color="tab:green", linestyle=":", label="rho0^2")
ax.plot(r_grid*k0, C_L_coherent, label="C_coherent")
ax.set_xlim(0, min(r_max*k0, 120))
ax.set_xlabel("r k0")
ax.set_ylabel("correlation")
ax.legend()
fig.tight_layout()
fig.savefig(output_dir / "diagnostics_Ccoherent_tail.png", dpi=180)
plt.show()

tail_mask = (r_grid >= r_tail_min) & (r_grid <= r_tail_max)
fig, ax = plt.subplots()
ax.plot(r_grid[tail_mask]*k0, (r_grid*C_L_coherent)[tail_mask], label="r C_coherent")
ax.plot(r_grid[tail_mask]*k0, (r_grid**2*C_L_coherent)[tail_mask], label="r^2 C_coherent")
ax.plot(r_grid[tail_mask]*k0, (r_grid**3*C_L_coherent)[tail_mask], label="r^3 C_coherent")
ax.set_xlabel("r k0")
ax.set_ylabel("tail-weighted correlation")
ax.legend()
fig.tight_layout()
fig.savefig(output_dir / "diagnostics_Ccoherent_tail_weighted.png", dpi=180)
plt.show()

In [ ]:
W_tail_only = ris.tail_only_window(r_grid, r_tail_min, r_tail_max)
H_tail = ris.hankel_transform(r_grid, C_L_coherent * W_tail_only, Q_grid)
H_window = ris.hankel_transform(r_grid, W_tail, Q_grid)
H_r2window = 4*np.pi * np.array([
    simpson(r_grid**2 * W_tail * np.sinc(Q*r_grid/np.pi), x=r_grid)
    for Q in Q_grid
])

fig, ax = plt.subplots()
ax.loglog(Q_grid/k0, np.abs(I_Q_raw), label="|H_coherent|")
ax.loglog(Q_grid/k0, np.abs(H_tail), label="|H_tail|")
ax.loglog(Q_grid/k0, np.abs(I_plateau_windowed), label="|H_plateau|")
ax.loglog(Q_grid/k0, np.abs(H_r2window), label="|H_window|")
ax.axvline(1, color="black", linestyle=":")
ax.axvline(2, color="black", linestyle="--")
ax.set_xlabel("Q/k0")
ax.set_ylabel("absolute transform")
ax.legend()
fig.tight_layout()
fig.savefig(output_dir / "diagnostics_window_transforms.png", dpi=180)
plt.show()

fig, ax = plt.subplots()
ax.semilogx(Q_grid/k0, Q_grid*I_Q_raw, label="Q H_coherent")
ax.semilogx(Q_grid/k0, Q_grid*H_tail, label="Q H_tail")
ax.axvline(1, color="black", linestyle=":")
ax.axvline(2, color="black", linestyle="--")
ax.axhline(k0**2/3, color="tab:orange", linestyle="--", label="k0^2/3")
ax.set_xlabel("Q/k0")
ax.set_ylabel("Q I(Q)")
ax.legend()
fig.tight_layout()
fig.savefig(output_dir / "diagnostics_window_transforms_QI.png", dpi=180)
plt.show()

In [ ]:
window_comparison = ris.compute_window_comparison(
    r_grid,
    C_L_coherent,
    Q_grid,
    window_types=("cosine", "hann_tail", "tukey", "exponential", "none"),
    taper_fractions=(0.50, 0.65, 0.80, 0.90),
    r_max=r_max,
)

fig, ax = plt.subplots()
for item in window_comparison:
    label = item["window_type"] if item["window_type"] == "none" else f"{item['window_type']} {item['taper_fraction']:.2f}"
    ax.loglog(Q_grid/k0, np.abs(item["I"]), label=label)
ax.axvline(1, color="black", linestyle=":")
ax.axvline(2, color="black", linestyle="--")
ax.set_xlabel("Q/k0")
ax.set_ylabel("|I_coherent(Q)|")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(output_dir / "diagnostics_window_comparison.png", dpi=180)
plt.show()

rmax_comparison = ris.compute_rmax_comparison(
    r_grid,
    C_L_coherent,
    Q_grid,
    k0,
    r_max_values=(100/k0, 150/k0, 200/k0, 300/k0),
    taper_fraction=0.75,
)

fig, ax = plt.subplots()
for item in rmax_comparison:
    ax.loglog(Q_grid/k0, np.abs(item["I"]), label=f"rmax*k0={item['r_max_over_k0']:.0f}")
ax.axvline(1, color="black", linestyle=":")
ax.axvline(2, color="black", linestyle="--")
ax.set_xlabel("Q/k0")
ax.set_ylabel("|I_coherent(Q)|")
ax.legend()
fig.tight_layout()
fig.savefig(output_dir / "diagnostics_rmax_comparison.png", dpi=180)
plt.show()

fig, ax = plt.subplots()
for item in rmax_comparison:
    ax.semilogx(Q_grid/k0, Q_grid*item["I"], label=f"rmax*k0={item['r_max_over_k0']:.0f}")
ax.axhline(k0**2/3, color="black", linestyle="--", label="k0^2/3")
ax.axvline(1, color="black", linestyle=":")
ax.axvline(2, color="black", linestyle="--")
ax.set_xlabel("Q/k0")
ax.set_ylabel("Q I_coherent(Q)")
ax.legend()
fig.tight_layout()
fig.savefig(output_dir / "diagnostics_rmax_comparison_QI.png", dpi=180)
plt.show()

In [ ]:
ingredient_transforms = ris.simplified_function_transforms(r_grid, Q_grid, k0, W_tail)

fig, ax = plt.subplots()
for name in ["g", "g2", "gp2", "gpp2", "bg", "cg"]:
    ax.loglog(Q_grid/k0, np.abs(ingredient_transforms[name]), label=name)
ax.axvline(1, color="black", linestyle=":")
ax.axvline(2, color="black", linestyle="--")
ax.set_xlabel("Q/k0")
ax.set_ylabel("abs(H[f](Q))")
ax.legend()
fig.tight_layout()
fig.savefig(output_dir / "diagnostics_simplified_function_transforms.png", dpi=180)
plt.show()

MJ_tail = M_J - 4*a*a
tail_projection_MJ = ris.tail_frequency_projection(
    r_grid,
    MJ_tail,
    k0,
    r_tail_min=r_tail_min,
    r_tail_max=r_tail_max,
)
print("M_J tail projection")
print(f"  RelAmp1 @ k0   = {tail_projection_MJ['RelAmp1']:.6g}")
print(f"  RelAmp2 @ 2*k0 = {tail_projection_MJ['RelAmp2']:.6g}")

fig, ax = plt.subplots()
ax.plot(r_grid*k0, M_J, label="M_J")
ax.axhline(4*a*a, color="tab:green", linestyle="--", label="4 a^2")
ax.set_xlabel("r k0")
ax.set_ylabel("M_J")
ax.legend()
fig.tight_layout()
fig.savefig(output_dir / "diagnostics_MJ.png", dpi=180)
plt.show()

fig, ax = plt.subplots()
ax.plot(r_grid[tail_mask]*k0, MJ_tail[tail_mask], label="M_J - 4a^2")
ax.plot(r_grid[tail_mask]*k0, (r_grid*MJ_tail)[tail_mask], label="r(M_J - 4a^2)")
ax.plot(r_grid[tail_mask]*k0, (r_grid**2*MJ_tail)[tail_mask], label="r^2(M_J - 4a^2)")
ax.set_xlabel("r k0")
ax.set_ylabel("tail-weighted M_J")
ax.legend()
fig.tight_layout()
fig.savefig(output_dir / "diagnostics_MJ_tail.png", dpi=180)
plt.show()

print("Diagnostic summary")
print(f"rho0 = {rho0:.12g}")
print(f"rho0^2 = {rho0**2:.12g}")
print(f"C_L[-1] = {C_L[-1]:.12g}")
print(f"C_coherent[-1] = {C_L_coherent[-1]:.12g}")
print(f"M_J[-1] = {M_J[-1]:.12g}")
print(f"expected M_J(infinity) = {4*a*a:.12g}")
print(f"C_coherent RelAmp1 = {tail_projection_coherent['RelAmp1']:.6g}")
print(f"C_coherent RelAmp2 = {tail_projection_coherent['RelAmp2']:.6g}")
print(f"M_J tail RelAmp1 = {tail_projection_MJ['RelAmp1']:.6g}")
print(f"M_J tail RelAmp2 = {tail_projection_MJ['RelAmp2']:.6g}")

q1_idx = int(np.argmin(np.abs(Q_grid/k0 - 1.0)))
q1_value = Q_grid[q1_idx] / k0
window_q1 = np.array([item["I"][q1_idx] for item in window_comparison], dtype=float)
rmax_q1 = np.array([item["I"][q1_idx] for item in rmax_comparison], dtype=float)
print(f"Q/k0 sample nearest 1 = {q1_value:.6g}")
print(f"window/taper |I_coherent(Q~k0)| min = {np.min(np.abs(window_q1)):.6g}, max = {np.max(np.abs(window_q1)):.6g}")
print(f"r_max |I_coherent(Q~k0)| min = {np.min(np.abs(rmax_q1)):.6g}, max = {np.max(np.abs(rmax_q1)):.6g}")
print("Inspect the saved comparison curves for stability under window type, taper start, and r_max.")

## Window Convergence Checks

These reuse the same computed `C_L(r)` and vary only the finite transform window. Strong changes in a feature with `r_max` or taper start indicate window sensitivity.

In [ ]:
convergence_curves = []
for r_max_test in r_max_convergence_values:
    mask = r_grid <= r_max_test
    if np.count_nonzero(mask) < 8:
        continue
    r_sub = r_grid[mask]
    C_sub = C_L[mask]
    for frac in taper_start_fractions:
        taper_start_test = frac * float(r_sub[-1])
        d = ris.compute_transform_diagnostics(
            r_sub,
            C_sub,
            Q_grid,
            k0,
            r_taper_start=taper_start_test,
            use_singular_split=use_singular_split,
            rc=rc,
        )
        convergence_curves.append((float(r_sub[-1]), frac, d["I_total"]))

fig, ax = plt.subplots()
for rmax_i, frac, curve in convergence_curves:
    ax.loglog(Q_grid/k0, np.abs(curve), label=f"rmax={rmax_i:g}, taper={frac:.2f}")
ax.set_xlabel("Q/k0")
ax.set_ylabel("|I_total(Q)|")
ax.legend(fontsize=8)
plt.show()

fig, ax = plt.subplots()
for rmax_i, frac, curve in convergence_curves:
    ax.semilogx(Q_grid/k0, Q_grid * curve, label=f"rmax={rmax_i:g}, taper={frac:.2f}")
ax.axhline(k0**2/3.0, color="black", linestyle="--", label="k0^2/3")
ax.set_xlabel("Q/k0")
ax.set_ylabel("Q I_total(Q)")
ax.legend(fontsize=8)
plt.show()

## Inverse Transform Check

The inverse radial 3D transform is `C(r) = (1/(2*pi^2)) int Q^2 I(Q) sinc(Q*r/pi) dQ`. Since `I_Q` was computed from the windowed, background-subtracted correlation, the inverse should be compared to `C_L_transform`, not the unwindowed full `C_L`.

In [ ]:
r_inverse_max = min(r_max, 200.0 / k0)
r_inverse = np.linspace(r_min, r_inverse_max, 1200)

C_inverse = np.empty_like(r_inverse)
weighted_I = Q_grid**2 * I_Q
for idx, r_value in enumerate(r_inverse):
    kernel = np.sinc(Q_grid * r_value / np.pi)
    C_inverse[idx] = (1.0 / (2.0 * np.pi**2)) * simpson(weighted_I * kernel, x=Q_grid)

C_inverse_input = C_L_sing + C_L_transform if use_singular_split else C_L_transform
C_target = np.interp(r_inverse, r_grid, C_inverse_input)
C_coherent_ref = np.interp(r_inverse, r_grid, C_L_coherent)

fig, ax = plt.subplots()
ax.semilogx(r_inverse, C_inverse, linewidth=2.0, label="inverse FT of I(Q)")
# ax.semilogx(r_inverse, C_target, alpha=0.75, label="C_L transform input")
# ax.semilogx(r_inverse, C_coherent_ref, alpha=0.35, label="C_L - rho0^2")
ax.axhline(0.0, color="black", linewidth=0.8)
ax.set_xlabel("r")
ax.set_ylabel("C(r)")
ax.legend()
plt.show()

fig, ax = plt.subplots()
ax.loglog(r_inverse, np.abs(C_inverse), linewidth=2.0, label="|inverse FT of I(Q)|")
ax.loglog(r_inverse, np.abs(C_target), alpha=0.75, label="|C_L transform input|")
ax.set_xlabel("r")
ax.set_ylabel("absolute value")
ax.legend()
plt.show()

## Save Demo Outputs

In [ ]:
output_dir = Path("rw_line_scattering_demo_output")
output_dir.mkdir(exist_ok=True)

np.savez(
    output_dir / "demo_data.npz",
    r_grid=r_grid,
    Q_grid=Q_grid,
    M_J=M_J,
    C_L=C_L,
    C_L_coherent=C_L_coherent,
    C_L_sing=C_L_sing,
    C_L_reg=C_L_reg,
    C_L_transform=C_L_transform,
    C_L_transform_raw=C_L_transform_raw,
    W_tail=W_tail,
    I_Q=I_Q,
    I_Q_raw=I_Q_raw,
    I_Q_full_direct=I_Q_full_direct,
    I_plateau_windowed=I_plateau_windowed,
    I_sing=I_sing,
    I_reg_windowed=I_reg_windowed,
    r_inverse=r_inverse,
    C_inverse=C_inverse,
    C_inverse_target=C_target,
    rho0=rho0,
    k0=k0,
    tail_start=tail_start,
    rc=rc,
    use_singular_split=use_singular_split,
    N_samp=N_samp,
    use_qmc=use_qmc,
    random_seed=random_seed,
)

ris.save_plots(output_dir, r_grid, Q_grid, k0, M_J, C_L, I_Q)
print(f"saved demo outputs to {output_dir.resolve()}")

for name in [
    "covariance_derivatives.png",
    "MJ_vs_r.png",
    "CL_loglog.png",
    "CL_coherent.png",
    "IQ_loglog.png",
    "QIQ_plateau.png",
    "diagnostics_Ccoherent_tail.png",
    "diagnostics_tail_frequency_projection.png",
    "diagnostics_window_transforms.png",
    "diagnostics_window_comparison.png",
    "diagnostics_rmax_comparison.png",
    "diagnostics_simplified_function_transforms.png",
    "diagnostics_MJ_tail.png",
]:
    print(name)
    display(Image(filename=str(output_dir / name)))